# OpenPKFlow Demo

This notebook demonstrates the core dissolution similarity workflow using the published package.
Everything runs from the pip-installed package — no local src/ imports.

In [ ]:
!pip install openpkflow --quiet

## 1. Quick f1/f2 from arrays

In [ ]:
from openpkflow.dissolution import f1, f2

reference = [20.0, 40.0, 60.0, 80.0, 90.0]
test      = [21.0, 39.0, 61.0, 79.0, 88.0]

print(f"f1 = {f1(reference, test):.2f}  (<=15 acceptable)")
print(f"f2 = {f2(reference, test):.2f}  (>=50 similar)")

## 2. Regulatory f2 method (FDA 85% rule)

When profiles approach complete dissolution, FDA guidance allows only one timepoint above 85% for both profiles.
Pass `method='regulatory'` to trim automatically.

In [ ]:
ref_late = [20.0, 40.0, 60.0, 80.0, 90.0, 95.0]
tst_late = [20.0, 40.0, 60.0, 80.0, 92.0, 96.0]

print(f"f2 all_points:  {f2(ref_late, tst_late, method='all_points'):.2f}")
print(f"f2 regulatory:  {f2(ref_late, tst_late, method='regulatory'):.2f}")
print("(regulatory trims the second above-85 point before computing)")

## 3. Bootstrap f2 confidence interval

For small samples (<12 vessels), use bootstrap f2 to get a confidence interval.
Each row is one vessel, each column is one timepoint.

In [ ]:
import numpy as np
from openpkflow.dissolution import bootstrap_f2

rng = np.random.default_rng(42)

ref_vessels = np.array([
    [20, 38, 56, 70, 83, 92],
    [19, 37, 55, 69, 82, 91],
    [21, 39, 57, 71, 84, 93],
    [20, 38, 56, 70, 83, 92],
    [19, 37, 55, 69, 82, 91],
    [21, 40, 58, 72, 85, 93],
], dtype=float)

tst_vessels = ref_vessels + rng.normal(0, 1.5, ref_vessels.shape)
tst_vessels = tst_vessels.clip(0, 100)

result = bootstrap_f2(ref_vessels, tst_vessels, n_replicates=5000, confidence_level=0.90, seed=42)
print(result.summary())

## 4. Load a CSV dataset and compare formulations

In [ ]:
from openpkflow.dissolution import DissolutionStudy
from openpkflow.datasets import example_dissolution_path, example_similar_path, example_not_similar_path

# Borderline similar (f2 ~58)
study = DissolutionStudy.from_csv(example_dissolution_path())
result = study.compare(reference="reference", test="test")
print(result.summary())

## 5. Compare all three example datasets side by side

In [ ]:
import pandas as pd

datasets = {
    "Borderline similar": example_dissolution_path(),
    "Clearly similar":    example_similar_path(),
    "Not similar":        example_not_similar_path(),
}

rows = []
for label, path in datasets.items():
    s = DissolutionStudy.from_csv(path)
    r = s.compare(reference="reference", test="test")
    rows.append({"Dataset": label, "f1": round(r.f1_value, 2), "f2": round(r.f2_value, 2), "Similar": r.f2_value >= 50})

pd.DataFrame(rows)

## 6. Generate an HTML report

In [ ]:
from pathlib import Path

study = DissolutionStudy.from_csv(example_dissolution_path())
result = study.compare(reference="reference", test="test")

report_path = Path("dissolution_report.html")
result.report(report_path, format="html")

print(f"Report saved to {report_path.resolve()}")
print("Open it in a browser to see the full regulatory-style report with the dissolution profile plot.")

## 7. Profile plot inline

In [ ]:
import matplotlib
matplotlib.use("Agg")  # headless — swap to inline if running in classic Jupyter
import matplotlib.pyplot as plt
import numpy as np

study = DissolutionStudy.from_csv(example_dissolution_path())
result = study.compare(reference="reference", test="test")

tp = np.array(result.time_points)
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(tp, result.reference_mean, "o-", color="#003366", label="Reference")
ax.plot(tp, result.test_mean,      "s--", color="#cc3300", label="Test")
ax.axhline(85, color="#888", linestyle=":", label="85% threshold")
ax.set_xlabel("Time (min)")
ax.set_ylabel("Mean % Dissolved")
ax.set_title(f"f1={result.f1_value:.1f}  f2={result.f2_value:.1f}")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("profile_plot.png", dpi=110)
print("Saved profile_plot.png")
plt.show()